In [19]:
import os
from pathlib import Path

IN_KAGGLE = os.path.exists('/kaggle/input')

if IN_KAGGLE:
    def find_data_dir(root, max_depth=4):
        for item in root.rglob('*'):
            if item.is_dir() and (item / 'data').exists():
                return item
            if len(item.parts) - len(root.parts) >= max_depth:
                continue
        return None

    project_folder = find_data_dir(Path('/kaggle/input'))
    if not project_folder:
        raise FileNotFoundError("Cannot find project with 'data' directory")
    print(f"Project: {project_folder}")

    missing = [pkg for pkg, imp in [
        ('sentence-transformers', 'sentence_transformers'),
        ('bertopic', 'bertopic'),
        ('hdbscan', 'hdbscan'),
        ('umap-learn', 'umap')
    ] if not __import__('importlib').util.find_spec(imp)]

    if missing:
        os.system(f'pip install -q {" ".join(missing)}')

    import torch
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
    else:
        print("Using CPU")
else:
    print("Running locally")
    project_folder = None

Project: /kaggle/input/datasets/anonymous/fyp-final/Final Year Project FYP
GPU: Tesla T4 (15.6 GB)


In [20]:
import sys
import os
from pathlib import Path

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import warnings
warnings.filterwarnings('ignore')

if IN_KAGGLE:
    src_path = project_folder / 'final_pipeline' / 'src'
    sys.path.insert(0, str(src_path))
    
    DATA_ROOT = project_folder / 'data'
    OUTPUT_DIR = Path('/kaggle/working/outputs')
    PROCESSED_DIR = Path('/kaggle/working/processed')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    EMBEDDINGS_CACHE_DIR = project_folder / 'final_pipeline' / 'outputs' / 'embeddings_cache'

else:
    sys.path.insert(0, str(Path.cwd()))
    DATA_ROOT = Path('../../data')
    OUTPUT_DIR = Path('../outputs')
    PROCESSED_DIR = DATA_ROOT / 'processed'
    EMBEDDINGS_CACHE_DIR = OUTPUT_DIR / 'embeddings_cache'

import pandas as pd
import logging

logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print(f"Data root: {DATA_ROOT}")

print(f"Processed: {PROCESSED_DIR}")
print(f"Output: {OUTPUT_DIR}")

Data root: /kaggle/input/datasets/anonymous/fyp-final/Final Year Project FYP/data
Processed: /kaggle/working/processed
Output: /kaggle/working/outputs


# BERTopic Pipeline - Computing Discourse Analysis

This notebook demonstrates the complete modular pipeline for both eras:
1. Text preprocessing (cleaning, lemmatization, stopword removal)
2. Load preprocessed data
3. Apply semantic chunking for long documents
4. Run BERTopic modeling with era-specific configurations
5. Aggregate results back to document level
6. Export results and visualizations

The pipeline runs for both Era1 and Era2 sequentially, each with its own configuration.

## Configuration

Select which eras to process and run each step independently.

In [21]:
from preprocessing.config import load_preprocessing_config
from preprocessing.text_preprocessor import TextPreprocessor
from preprocessing.data_loader import DataLoader
from topic_modelling.utils.data_utils import load_processed_data
from topic_modelling.utils.chunker import SemanticChunker
from topic_modelling.core.topic_modeler import TopicModeler
from config import load_config

# Initialize preprocessing components
preprocess_config = load_preprocessing_config()
preprocessor = TextPreprocessor(preprocess_config)
cleaned_root = DATA_ROOT / 'cleaned'
data_loader = DataLoader(cleaned_root)

print("All modules imported successfully")

All modules imported successfully


In [22]:
# Select which eras to process
eras = [1, 2]  # Change to [1] or [2] to process only one era

## Step 1: Text Preprocessing (Optional)

Run this step only if you need to preprocess the raw data. Skip if preprocessed data already exists.

In [23]:
# import nltk
# nltk.download('wordnet', quiet=True)
# nltk.download('omw-1.4', quiet=True)
# nltk.download('punkt', quiet=True)
# nltk.download('punkt_tab', quiet=True)
# nltk.download('stopwords', quiet=True)

# for era in eras:
#     print(f"\nProcessing Era {era}")
    
#     logger.info(f"Loading raw cleaned data for Era{era}...")
#     raw_texts, raw_metadata = data_loader.load_era_data(f'Era{era}')
#     print(f"Loaded {len(raw_texts)} raw documents")
    
#     logger.info(f"Applying preprocessing...")
#     processed_texts = []
#     processed_metadata = []
#     skipped = 0
    
#     for text, metadata in zip(raw_texts, raw_metadata):
#         try:
#             processed = preprocessor.preprocess_text(text)
#             if len(processed.split()) < preprocess_config.min_doc_length:
#                 skipped += 1
#                 continue
#             processed_texts.append(processed)
#             processed_metadata.append(metadata)
#         except Exception as e:
#             logger.warning(f"Preprocessing failed for {metadata.get('filepath', 'unknown')}: {e}")
#             skipped += 1
    
#     print(f"Preprocessed {len(processed_texts)} documents ({skipped} skipped)")
    
#     preprocessor.save_processed_texts(processed_texts, processed_metadata, str(PROCESSED_DIR.parent), f'Era{era}')
#     print(f"Saved to {PROCESSED_DIR / f'Era{era}'}")

## Step 2: Load Preprocessed Data

In [24]:
texts_dict = {}
metadata_dict = {}

for era in eras:
    logger.info(f"Loading preprocessed data for Era{era}...")
    texts, metadata_list = load_processed_data(PROCESSED_DIR.parent, era)
    texts_dict[era] = texts
    metadata_dict[era] = metadata_list
    print(f"Loaded {len(texts)} preprocessed documents for Era{era}")

INFO - Loading preprocessed data for Era1...
INFO - Loaded 165 processed documents from Era 1
INFO - Loading preprocessed data for Era2...
INFO - Loaded 181 processed documents from Era 2


Loaded 165 preprocessed documents for Era1
Loaded 181 preprocessed documents for Era2


## Step 3: Semantic Chunking

In [25]:
# Apply semantic chunking to documents
chunked_texts_dict = {}
chunked_metadata_dict = {}

chunker = SemanticChunker(
    max_tokens=5000,
    overlap_tokens=200,
    chunk_threshold=7000
)

for era in eras:
    logger.info(f"Applying semantic chunking for Era{era}...")
    chunked_texts, chunked_metadata = chunker.chunk_documents(texts_dict[era], metadata_dict[era])
    chunked_texts_dict[era] = chunked_texts
    chunked_metadata_dict[era] = chunked_metadata
    print(f"Created {len(chunked_texts)} chunks from {len(texts_dict[era])} documents for Era{era}")

INFO - Applying semantic chunking for Era1...
INFO - Chunked Era1/HistoricalReflections/historical_reflections_03.txt (12400 tokens) into 3 chunks
INFO - Chunked Era1/HistoricalReflections/historical_reflections_07.txt (8616 tokens) into 2 chunks
INFO - Chunked Era1/CACM/cacm_1966_08/section_02.txt (8637 tokens) into 2 chunks
INFO - Chunked Era1/CACM/cacm_1969_08/section_03.txt (8404 tokens) into 2 chunks
INFO - Chunked Era1/Academic Papers/academic_papers_05.txt (18682 tokens) into 4 chunks
INFO - Chunked Era1/Academic Papers/academic_papers_06.txt (9205 tokens) into 2 chunks
INFO - Chunked Era1/Academic Papers/academic_papers_07.txt (12920 tokens) into 3 chunks
INFO - Chunked Era1/Academic Papers/academic_papers_08.txt (10799 tokens) into 3 chunks
INFO - Chunked Era1/Interviews/interview_01.txt (8378 tokens) into 2 chunks
INFO - Chunked Era1/Interviews/interview_03.txt (23453 tokens) into 5 chunks
INFO - Chunked Era1/Interviews/interview_04.txt (8446 tokens) into 2 chunks
INFO - Chun

Created 212 chunks from 165 documents for Era1


INFO - Chunked Era2/Papers/papers_05.txt (9620 tokens) into 2 chunks
INFO - Chunked Era2/Papers/papers_06.txt (9620 tokens) into 2 chunks
INFO - Chunked Era2/Papers/papers_32.txt (8144 tokens) into 2 chunks
INFO - Chunked Era2/Papers/papers_08.txt (7187 tokens) into 2 chunks
INFO - Chunked Era2/Papers/papers_22.txt (11194 tokens) into 3 chunks
INFO - Chunked Era2/Papers/papers_31.txt (10050 tokens) into 2 chunks
INFO - Chunked Era2/Papers/papers_25.txt (14454 tokens) into 3 chunks
INFO - Chunked Era2/Wired/wired_50.txt (7092 tokens) into 2 chunks
INFO - Chunking: 181 docs -> 198 chunks


Created 198 chunks from 181 documents for Era2


## Step 4: BERTopic Modeling

Train topic models for each era using era-specific configurations.

In [26]:
# Pre-compute embeddings
from sentence_transformers import SentenceTransformer
import numpy as np

embeddings_dict = {}
CACHE_DIR = EMBEDDINGS_CACHE_DIR
CACHE_DIR.mkdir(parents=True, exist_ok=True)

for era in eras:
    print(f"\nProcessing embeddings for Era {era}")

    config = load_config(era=era)
    embedding_model_name = config.embedding_model
    cache_model_slug = embedding_model_name.replace('/', '_').replace('-', '_')
    cache_path = CACHE_DIR / f'era{era}_embeddings_{cache_model_slug}.npy'

    if cache_path.exists():
        embeddings = np.load(cache_path)
        # Invalidate if mostly zero-vectors (leftover from prior GPU NaN bug)
        if np.all(embeddings == 0, axis=1).sum() > len(embeddings) // 2:
            print(f"Cached embeddings invalid (mostly zeros), regenerating...")
            cache_path.unlink()
        else:
            embeddings_dict[era] = embeddings
            print(f"Loaded {embeddings.shape[0]} embeddings (dim={embeddings.shape[1]})\n")
            continue

    # gte-base-en-v1.5 custom code (NewEmbeddings + NTKScalingRotaryEmbedding)
    # produces NaN on CUDA via its RoPE gather ops + PyTorch 2.9 interaction.
    # Encode on CPU where it works correctly; cached after first run.
    print(f"Loading model (CPU): {embedding_model_name}")
    st_model = SentenceTransformer(embedding_model_name, trust_remote_code=True, device='cpu')

    chunks = chunked_texts_dict[era]
    print(f"Encoding {len(chunks)} chunks on CPU (one-time, result is cached)...")

    embeddings = st_model.encode(
        chunks,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    nan_rows = np.isnan(embeddings).any(axis=1)
    if nan_rows.any():
        print(f"⚠️  {nan_rows.sum()} NaN embeddings replaced with zeros")
        embeddings[nan_rows] = 0.0

    embeddings_dict[era] = embeddings
    np.save(cache_path, embeddings)
    del st_model
    print(f"✓ Generated {embeddings.shape[0]} embeddings (dim={embeddings.shape[1]})\n")


Processing embeddings for Era 1
Loaded 212 embeddings (dim=768)


Processing embeddings for Era 2
Loaded 198 embeddings (dim=768)



In [27]:
# from config import load_config

# config_era1 = load_config(era=1)

# # Override params — tweak these values and re-run the training cell
# config_era1.hdbscan.min_cluster_size = 15
# config_era1.hdbscan.min_samples = 5
# config_era1.umap.n_neighbors = 15
# config_era1.umap.n_components = 10
# config_era1.umap.min_dist = 0.0

# # Inject into the training loop by monkey-patching load_config
# import config as _config_module
# _orig_load = _config_module.load_config
# _config_module.load_config = lambda era: config_era1 if era == 1 else _orig_load(era=era)
# print("Era 1 config overridden")

In [28]:
# Train BERTopic models for each era
topics_dict = {}
probs_dict = {}
modelers_dict = {}

for era in eras:
    print(f"\nTraining BERTopic model for Era {era}")

    config = load_config(era=era)
    if (era ==1):
        config.hdbscan.min_cluster_size = 5
        config.hdbscan.min_samples = 2
        config.umap.n_neighbors =12
        config.umap.n_components = 20
        config.umap.min_dist = .1

        
    print(f"Config: min_cluster_size={config.hdbscan.min_cluster_size}, min_samples={config.hdbscan.min_samples}, n_components={config.umap.n_components}")

    logger.info(f"Training model with pre-computed embeddings...")
    modeler = TopicModeler(config)
    topics, probs = modeler.fit_transform(
        chunked_texts_dict[era],
        embeddings=embeddings_dict[era],
        metadata=chunked_metadata_dict[era]
    )

    topics_dict[era] = topics
    probs_dict[era] = probs
    modelers_dict[era] = modeler

    print(f"Identified {len(set(topics))} topics (including outliers) for Era {era}\n")

INFO - Training model with pre-computed embeddings...
INFO - Use pytorch device_name: cuda:0
INFO - Load pretrained SentenceTransformer: Alibaba-NLP/gte-base-en-v1.5



Training BERTopic model for Era 1
Config: min_cluster_size=5, min_samples=2, n_components=20


Loading weights:   0%|          | 0/135 [00:00<?, ?it/s]

INFO - Fitting BERTopic model...
2026-02-19 22:47:54,302 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-19 22:47:54,972 - BERTopic - Dimensionality - Completed ✓
2026-02-19 22:47:54,973 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-19 22:47:54,996 - BERTopic - Cluster - Completed ✓
2026-02-19 22:47:54,999 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-19 22:47:56,699 - BERTopic - Representation - Completed ✓
INFO - Training model with pre-computed embeddings...
INFO - Use pytorch device_name: cuda:0
INFO - Load pretrained SentenceTransformer: Alibaba-NLP/gte-base-en-v1.5


Identified 16 topics (including outliers) for Era 1


Training BERTopic model for Era 2
Config: min_cluster_size=3, min_samples=5, n_components=10


Loading weights:   0%|          | 0/135 [00:00<?, ?it/s]

INFO - Fitting BERTopic model...
2026-02-19 22:47:59,504 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-19 22:47:59,910 - BERTopic - Dimensionality - Completed ✓
2026-02-19 22:47:59,911 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-19 22:47:59,928 - BERTopic - Cluster - Completed ✓
2026-02-19 22:47:59,931 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-19 22:48:01,021 - BERTopic - Representation - Completed ✓


Identified 15 topics (including outliers) for Era 2



## Step 5: Aggregate to Document Level

In [29]:
# Aggregate chunk-level topics to document level with multi-topic support
doc_topics_dict = {}

for era in eras:
    logger.info(f"Aggregating chunk topics to document level for Era {era}...")

    config = load_config(era=era)

    multi_topic_enabled = config.multi_topic_assignment.enabled
    threshold = config.multi_topic_assignment.probability_threshold

    topics = topics_dict[era]
    probs = probs_dict[era]

    # Group chunks by document
    doc_topic_probs = {}
    for i, metadata in enumerate(chunked_metadata_dict[era]):
        doc_id = metadata.get('doc_id', metadata.get('filepath', f'doc_{i}'))

        if doc_id not in doc_topic_probs:
            doc_topic_probs[doc_id] = {}

        if probs is not None and len(probs.shape) == 2:
            for topic_id, prob in enumerate(probs[i]):
                if topic_id not in doc_topic_probs[doc_id]:
                    doc_topic_probs[doc_id][topic_id] = []
                doc_topic_probs[doc_id][topic_id].append(prob)
        else:
            topic = topics[i]
            max_prob = probs[i] if probs is not None and len(probs.shape) == 1 else 1.0
            if topic not in doc_topic_probs[doc_id]:
                doc_topic_probs[doc_id][topic] = []
            doc_topic_probs[doc_id][topic].append(max_prob)

    # Average probabilities across chunks and filter by threshold
    doc_topics_list = []
    for doc_id, topic_prob_dict in doc_topic_probs.items():
        topic_avgs = {topic: np.mean(probs_list) for topic, probs_list in topic_prob_dict.items()}

        if multi_topic_enabled:
            valid_topics = {
                topic: prob for topic, prob in topic_avgs.items()
                if prob >= threshold and topic != -1
            }

            if not valid_topics:
                max_topic = max(topic_avgs.items(), key=lambda x: x[1])
                valid_topics = {max_topic[0]: max_topic[1]}
        else:
            max_topic = max(topic_avgs.items(), key=lambda x: x[1])
            valid_topics = {max_topic[0]: max_topic[1]}

        chunk_count = sum(1 for m in chunked_metadata_dict[era]
                         if m.get('doc_id', m.get('filepath')) == doc_id)

        for topic, prob in sorted(valid_topics.items(), key=lambda x: x[1], reverse=True):
            doc_topics_list.append({
                'doc_id': doc_id,
                'topic': topic,
                'probability': prob,
                'chunk_count': chunk_count,
                'num_topics': len(valid_topics)
            })

    doc_topics_df = pd.DataFrame(doc_topics_list)
    doc_topics_dict[era] = doc_topics_df

    multi_topic_docs = doc_topics_df[doc_topics_df['num_topics'] > 1]['doc_id'].nunique()
    total_docs = doc_topics_df['doc_id'].nunique()
    print(f"Aggregated topics for {total_docs} documents for Era {era}")
    if multi_topic_enabled:
        print(f"  Multi-topic enabled: {multi_topic_docs} documents have multiple topics ({multi_topic_docs/total_docs*100:.1f}%)")
        print(f"  Threshold: {threshold}")
    else:
        print(f"  Single-topic mode")

INFO - Aggregating chunk topics to document level for Era 1...
INFO - Aggregating chunk topics to document level for Era 2...


Aggregated topics for 165 documents for Era 1
  Multi-topic enabled: 10 documents have multiple topics (6.1%)
  Threshold: 0.15
Aggregated topics for 181 documents for Era 2
  Multi-topic enabled: 5 documents have multiple topics (2.8%)
  Threshold: 0.15


## Step 6: Export Results and Visualizations

In [30]:
# Export results and generate visualizations
for era in eras:
    print(f"\n{'='*70}")
    print(f"RESULTS FOR ERA {era}")
    print(f"{'='*70}")

    logger.info(f"Exporting results for Era {era}...")

    modeler = modelers_dict[era]
    topic_info = modeler.get_topic_info()
    topic_keywords = modeler.get_topic_keywords()

    print(f"\nTOPIC KEYWORDS (Top 5 words per topic):")
    print("-" * 70)

    for _, row in topic_keywords.iterrows():
        topic_id = row['Topic']
        keywords_list = row['Keywords'][:5]

        if topic_id == -1:
            topic_name = "Outliers"
        else:
            topic_name = f"Topic {topic_id}"

        keywords = ", ".join(keywords_list)

        doc_count = topic_info[topic_info['Topic'] == topic_id]['Count'].values
        count_str = f"({doc_count[0]} docs)" if len(doc_count) > 0 else ""

        print(f"{topic_name:15s} {count_str:12s}: {keywords}")

    outlier_info = topic_info[topic_info['Topic'] == -1]
    if not outlier_info.empty:
        count = outlier_info['Count'].values[0]
        print(f"{'Outliers':15s} ({count} docs): Documents that don't fit any topic")

    print("-" * 70)

    era_output_dir = OUTPUT_DIR / f'era{era}'
    era_output_dir.mkdir(parents=True, exist_ok=True)

    topic_info.to_csv(era_output_dir / f"Era{era}_topic_info.csv", index=False)
    topic_keywords.to_csv(era_output_dir / f"Era{era}_topic_keywords.csv", index=False)
    doc_topics_dict[era].to_csv(era_output_dir / f"Era{era}_document_topics.csv", index=False)

    print(f"\nSaved CSV files to {era_output_dir}")

    modeler.save_visualizations(era_output_dir, f'Era{era}')

    print(f"Generated visualizations\n")

print(f"\n{'='*70}")
print("PIPELINE COMPLETE")
print(f"{'='*70}")

INFO - Exporting results for Era 1...
INFO - Attempting to generate visualizations for Era1
INFO - Era1 has 15 topics (excluding outliers)
INFO - Generating topic visualization for Era1
INFO - Saved topic visualization for Era1
INFO - Generating topic hierarchy for Era1



RESULTS FOR ERA 1

TOPIC KEYWORDS (Top 5 words per topic):
----------------------------------------------------------------------
Topic 0         (41 docs)   : thing, would, people, think, one
Topic 1         (19 docs)   : program, function, system, computer, form
Topic 2         (17 docs)   : cobol, committee, language, computer, problem
Topic 3         (13 docs)   : program, letter, solution, zero, may
Topic 4         (11 docs)   : logic, system, language, programming, computer
Topic 5         (10 docs)   : cj, th, one, year, computer
Topic 6         (9 docs)    : gb, jb, program, yes, dijkstra
Topic 7         (9 docs)    : unit, format, algol, program, list
Topic 8         (9 docs)    : computer, fortran, programming, system, year
Topic 9         (9 docs)    : language, ieee, programming, tower, publication
Topic 10        (8 docs)    : cobol, language, noise, statement, data
Topic 11        (8 docs)    : newell, norberg, people, thing, one
Topic 12        (7 docs)    : uncol, sale

INFO - Saved topic hierarchy for Era1
INFO - Generating topic barchart for Era1
INFO - Saved topic barchart for Era1
INFO - Completed visualizations for Era1
INFO - Exporting results for Era 2...
INFO - Attempting to generate visualizations for Era2
INFO - Era2 has 14 topics (excluding outliers)
INFO - Generating topic visualization for Era2
INFO - Saved topic visualization for Era2
INFO - Generating topic hierarchy for Era2


Generated visualizations


RESULTS FOR ERA 2

TOPIC KEYWORDS (Top 5 words per topic):
----------------------------------------------------------------------
Topic 0         (42 docs)   : model, one, code, tool, say
Topic 1         (18 docs)   : code, test, codex, suggestion, quality
Topic 2         (15 docs)   : assistant, code, activity, student, response
Topic 3         (12 docs)   : chatgpt, participant, code, copilot, developer
Topic 4         (12 docs)   : code, experiment, developer, copilot, effect
Topic 5         (11 docs)   : code, jules, description, method, copilot
Topic 6         (10 docs)   : participant, canvas, design, chatgpt canvas, figure chatgpt
Topic 7         (10 docs)   : student, assistant, education, learning, programming
Topic 8         (8 docs)    : vibe, vibe coding, coding, agent, startup
Topic 9         (7 docs)    : copilot, user, participant, chatgpt, software
Topic 10        (7 docs)    : copilot, code, genai, http, software
Topic 11        (6 docs)    :

INFO - Saved topic hierarchy for Era2
INFO - Generating topic barchart for Era2
INFO - Saved topic barchart for Era2
INFO - Completed visualizations for Era2


Generated visualizations


PIPELINE COMPLETE


## Results Summary

View the outputs in `final_pipeline/outputs/era1/` and `final_pipeline/outputs/era2/`

## Step 6b: Sentiment Analysis (Optional)

In [31]:
# Define merged topics (optional) - leave empty {} to skip
MERGED_TOPICS = {
    # 'era_1': {'interviews': [1, 6]},

    # 'era_2': {'ai_agents': [2, 3, 5]}
}

## Quick Setup for Sentiment Analysis Only

Run this cell if you want to do sentiment analysis without re-running topic modeling. It loads:
- Preprocessed texts from `data/processed/Era{X}/`
- Previously generated topic assignments from `outputs/era{X}/Era{X}_document_topics.csv`

In [32]:
# Quick Setup for Sentiment Analysis (skip topic modelling cells if already run)
# Run cells 1-2 first for environment setup, then run this cell

from sentiment_analysis.utils import load_documents_for_sentiment
import pandas as pd

# Load preprocessed texts (needed for sentiment analysis)
texts_dict = {}
metadata_dict = {}
doc_ids_dict = {}
pub_dates_dict = {}

for load_era in eras:
    print(f"Loading data for Era {load_era} sentiment analysis...")
    texts, doc_ids, pub_dates, metadata_list = load_documents_for_sentiment(DATA_ROOT, load_era)
    texts_dict[load_era] = texts
    metadata_dict[load_era] = metadata_list
    doc_ids_dict[load_era] = doc_ids
    pub_dates_dict[load_era] = pub_dates
    print(f"  Loaded {len(texts)} documents")

# Load previously generated topic assignments
doc_topics_dict = {}

for load_era in eras:
    era_output_dir = OUTPUT_DIR / f'era{load_era}'
    topics_file = era_output_dir / f'Era{load_era}_document_topics.csv'

    if topics_file.exists():
        df = pd.read_csv(topics_file)
        original_len = len(df)

        # Filter to keep only doc_ids that belong to this era
        era_prefix = f'Era{load_era}/'
        df = df[df['doc_id'].str.startswith(era_prefix)]
        filtered_len = len(df)

        doc_topics_dict[load_era] = df

        if original_len != filtered_len:
            print(f"  Warning: Filtered out {original_len - filtered_len} rows from other eras")
        print(f"  Loaded {filtered_len} topic assignments for Era {load_era}")
    else:
        print(f"  Topic file not found: {topics_file}")
        print(f"  Run topic modelling first, or sentiment will skip topic aggregation")

print("\nReady for sentiment analysis")

Loading data for Era 1 sentiment analysis...


INFO - Loaded 165 documents for sentiment analysis from Era1
INFO - Documents with dates: 165


  Loaded 165 documents
Loading data for Era 2 sentiment analysis...


INFO - Loaded 181 documents for sentiment analysis from Era2
INFO - Documents with dates: 181


  Loaded 181 documents
  Loaded 175 topic assignments for Era 1
  Loaded 186 topic assignments for Era 2

Ready for sentiment analysis


In [33]:
from sentiment_analysis import SentimentAnalyzer, TopicSentimentAggregator
from sentiment_analysis.visualizations import create_all_visualizations, plot_temporal_sentiment_by_topic
import torch

# Download required NLTK data for sentiment analysis
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)

# Run sentiment analysis for each era
# Use GPU if available, larger batch size for speed
device = 0 if torch.cuda.is_available() else -1
batch_size = 16 if torch.cuda.is_available() else 8

print(f"Using device: {'GPU' if device == 0 else 'CPU'}")
sentiment_analyzer = SentimentAnalyzer(device=device)
aggregator = TopicSentimentAggregator()

for sent_era in eras:
    print(f"\n{'='*60}\nERA {sent_era} SENTIMENT ANALYSIS\n{'='*60}")

    era_output_dir = OUTPUT_DIR / f'era{sent_era}'

    doc_ids = doc_ids_dict[sent_era]
    pub_dates = pub_dates_dict[sent_era]

    print(f"Processing {len(texts_dict[sent_era])} documents...")

    sentiments = sentiment_analyzer.analyze_batch(texts_dict[sent_era], batch_size=batch_size)

    pd.DataFrame({
        'doc_id': doc_ids, 'publication_date': pub_dates,
        'sentiment_label': [s['label'] for s in sentiments],
        'sentiment_score': [s['score'] for s in sentiments],
        'compound_score': [s.get('compound_score', None) for s in sentiments],
        'prob_positive': [s.get('prob_positive', None) for s in sentiments],
        'prob_neutral': [s.get('prob_neutral', None) for s in sentiments],
        'prob_negative': [s.get('prob_negative', None) for s in sentiments]
    }).to_csv(era_output_dir / f'Era{sent_era}_document_sentiments.csv', index=False)

    topic_sentiments = aggregator.aggregate_by_topic(doc_topics_dict[sent_era], sentiments, doc_ids)
    topic_sentiments_temporal = aggregator.aggregate_by_topic_temporal(
        doc_topics_dict[sent_era], sentiments, doc_ids, pub_dates, time_period='year'
    )

    topic_sentiments.to_csv(era_output_dir / f'Era{sent_era}_topic_sentiments.csv', index=False)
    topic_sentiments_temporal.to_csv(era_output_dir / f'Era{sent_era}_topic_sentiments_temporal.csv', index=False)
    print(f"Saved sentiment CSVs")

    create_all_visualizations(sent_era, era_output_dir)
    print(f"Generated visualizations")

    era_key = f'era_{sent_era}'
    if era_key in MERGED_TOPICS and MERGED_TOPICS[era_key]:
        for name, topic_ids in MERGED_TOPICS[era_key].items():
            merged_docs = doc_topics_dict[sent_era][doc_topics_dict[sent_era]['topic'].isin(topic_ids)].copy()
            if merged_docs.empty:
                continue

            merged_docs['topic'] = name
            merged_temporal = aggregator.aggregate_by_topic_temporal(merged_docs, sentiments, doc_ids, pub_dates)
            merged_temporal.to_csv(era_output_dir / f'era{sent_era}_{name}_topic_sentiments_temporal.csv', index=False)
            print(f"  Saved merged topic '{name}' (topics {topic_ids})")

print("\n✓ Sentiment analysis complete for all eras")

INFO - Loading sentiment model: lxyuan/distilbert-base-multilingual-cased-sentiments-student
INFO - Device: GPU


Using device: GPU


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

INFO - Sentiment model loaded successfully
INFO - Analyzing 165 documents...



ERA 1 SENTIMENT ANALYSIS
Processing 165 documents...


INFO - Processed 100/165 documents
INFO - Completed sentiment analysis for 165 documents
INFO - Aggregating sentiments by topic...
INFO - Aggregated sentiments for 15 topics
INFO - Aggregating sentiments by topic and year...
INFO - Aggregated sentiments for 15 topics across 29 time periods
INFO - Creating sentiment visualizations for Era 1...
INFO - Creating temporal sentiment plot...
INFO - Saved temporal sentiment plot to /kaggle/working/outputs/era1/Era1_sentiment_temporal.html
INFO - Creating sentiment heatmap...
INFO - Saved sentiment heatmap to /kaggle/working/outputs/era1/Era1_sentiment_heatmap.html
INFO - Creating sentiment distribution plot...
INFO - Saved sentiment distribution plot to /kaggle/working/outputs/era1/Era1_sentiment_distribution.html
INFO - All visualizations created successfully for Era 1
INFO - Analyzing 181 documents...


Saved sentiment CSVs
Generated visualizations

ERA 2 SENTIMENT ANALYSIS
Processing 181 documents...


INFO - Processed 100/181 documents
INFO - Completed sentiment analysis for 181 documents
INFO - Aggregating sentiments by topic...
INFO - Aggregated sentiments for 14 topics
INFO - Aggregating sentiments by topic and year...
INFO - Aggregated sentiments for 14 topics across 9 time periods
INFO - Creating sentiment visualizations for Era 2...
INFO - Creating temporal sentiment plot...
INFO - Saved temporal sentiment plot to /kaggle/working/outputs/era2/Era2_sentiment_temporal.html
INFO - Creating sentiment heatmap...
INFO - Saved sentiment heatmap to /kaggle/working/outputs/era2/Era2_sentiment_heatmap.html
INFO - Creating sentiment distribution plot...
INFO - Saved sentiment distribution plot to /kaggle/working/outputs/era2/Era2_sentiment_distribution.html
INFO - All visualizations created successfully for Era 2


Saved sentiment CSVs
Generated visualizations

✓ Sentiment analysis complete for all eras


## Step 7: Cross-Era Similarity Analysis

Compute and visualize similarity between Era1 and Era2 documents based on their embeddings.

In [34]:
# Quick setup for similarity analysis only (run this instead of topic modelling cells)
from pathlib import Path

# Define cache directory and model name
CACHE_DIR = EMBEDDINGS_CACHE_DIR
embedding_model_name = 'Alibaba-NLP/gte-base-en-v1.5'

print(f"Cache directory: {CACHE_DIR}")
print(f"Embedding model: {embedding_model_name}")
print(f"\nExpected embedding files in: {CACHE_DIR.absolute()}")
print(f"  - era1_embeddings_Alibaba_NLP_gte_base_en_v1.5.npy")
print(f"  - era2_embeddings_Alibaba_NLP_gte_base_en_v1.5.npy")

Cache directory: /kaggle/input/datasets/anonymous/fyp-final/Final Year Project FYP/final_pipeline/outputs/embeddings_cache
Embedding model: Alibaba-NLP/gte-base-en-v1.5

Expected embedding files in: /kaggle/input/datasets/anonymous/fyp-final/Final Year Project FYP/final_pipeline/outputs/embeddings_cache
  - era1_embeddings_Alibaba_NLP_gte_base_en_v1.5.npy
  - era2_embeddings_Alibaba_NLP_gte_base_en_v1.5.npy


In [35]:
# Cross-Era Topic Similarity Analysis
# Generates 3 heatmaps: Era1 within, Era2 within, Cross-era

from cross_era_analysis.similarity_analyzer import EmbeddingSimilarityAnalyzer
from cross_era_analysis.visualizations import SimilarityVisualizer

# Initialize analyzer and load cached data
analyzer = EmbeddingSimilarityAnalyzer(CACHE_DIR, embedding_model_name)
analyzer.load_embeddings(era=1)
analyzer.load_embeddings(era=2)
analyzer.load_metadata(DATA_ROOT, era=1)
analyzer.load_metadata(DATA_ROOT, era=2)

# Generate all 3 topic similarity heatmaps
similarity_output_dir = OUTPUT_DIR / 'cross_era_analysis'
visualizer = SimilarityVisualizer(similarity_output_dir)
visualizer.create_all_heatmaps(analyzer, OUTPUT_DIR, top_n_matches=5)

print(f"\nTopic similarity analysis complete")

INFO - Loaded Era1: 212 embeddings (dim=768)
INFO - Loaded Era2: 198 embeddings (dim=768)
INFO - Loaded 165 processed documents from Era 1
INFO - Chunked Era1/Academic Papers/academic_papers_05.txt (18682 tokens) into 4 chunks
INFO - Chunked Era1/Academic Papers/academic_papers_06.txt (9205 tokens) into 2 chunks
INFO - Chunked Era1/Academic Papers/academic_papers_07.txt (12920 tokens) into 3 chunks
INFO - Chunked Era1/Academic Papers/academic_papers_08.txt (10799 tokens) into 3 chunks
INFO - Chunked Era1/CACM/cacm_1966_08/section_02.txt (8637 tokens) into 2 chunks
INFO - Chunked Era1/CACM/cacm_1969_08/section_03.txt (8404 tokens) into 2 chunks
INFO - Chunked Era1/HiringTrends/hiring_trends_02.txt (10799 tokens) into 3 chunks
INFO - Chunked Era1/HistoricalReflections/historical_reflections_03.txt (12400 tokens) into 3 chunks
INFO - Chunked Era1/HistoricalReflections/historical_reflections_07.txt (8616 tokens) into 2 chunks
INFO - Chunked Era1/Interviews/interview_01.txt (8378 tokens) in

Era 1 within-era: min=0.829, max=1.000
  ✓ Saved era1_within_similarity_matrix.csv
Era 2 within-era: min=0.768, max=1.000
  ✓ Saved era2_within_similarity_matrix.csv
Cross-era: min=0.345, max=0.576
  ✓ Saved era1_era2_cross_similarity_matrix.csv

TOP CROSS-ERA TOPIC MATCHES

Era1 Topic 0: thing, would, people, think, one
  1. Era2 T0 (0.464): model, one, code, tool, say
  2. Era2 T4 (0.443): code, experiment, developer, copilot, effect
  3. Era2 T5 (0.438): code, jules, description, method, copilot
  4. Era2 T6 (0.435): participant, canvas, design, chatgpt canvas, figure chatgpt
  5. Era2 T10 (0.431): copilot, code, genai, http, software

Era1 Topic 1: program, function, system, computer, form
  1. Era2 T0 (0.443): model, one, code, tool, say
  2. Era2 T5 (0.426): code, jules, description, method, copilot
  3. Era2 T4 (0.422): code, experiment, developer, copilot, effect
  4. Era2 T9 (0.419): copilot, user, participant, chatgpt, software
  5. Era2 T10 (0.414): copilot, code, genai, htt

In [36]:
import shutil

# Create zip files of outputs for easy download
if IN_KAGGLE:
    print("Creating zip files for download...")
    shutil.make_archive('/kaggle/working/all_outputs', 'zip', OUTPUT_DIR)
    print(f"Created /kaggle/working/all_outputs.zip")
    print("Download this file from the Output tab")

elif IN_COLAB:
    print("Creating zip files in Google Drive...")
    zip_base = project_folder.parent / 'FYP_outputs'
    shutil.make_archive(str(zip_base), 'zip', OUTPUT_DIR)
    print(f"Created {zip_base}.zip in Google Drive")

else:
    print("Running locally - outputs saved to:")
    print(f"  {OUTPUT_DIR.absolute()}")
    print(f"\nTo create a zip file, run:")
    print(f"  shutil.make_archive('outputs_backup', 'zip', r'{OUTPUT_DIR.absolute()}')")

Creating zip files for download...
Created /kaggle/working/all_outputs.zip
Download this file from the Output tab
